# Phase 2B — Weak-Pair Weighted Fine-tuning from Phase 1 1000-Pair Model

This notebook starts from the **Phase 1 1000-example balanced all-462 model** and continues training with weak-pair weighted sampling.

Design:
- Phase 1 input: `outputs/labse_phase1_all462_1000pairs_balanced/best_model`
- Training pairs: all 462 directed Indic→Indic pairs
- Examples per directed pair: 1000
- Weak directions are sampled more frequently using `WeightedRandomSampler`
- No English pivot is used
- Epoch-level checkpoint/resume is enabled


In [ ]:
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    _guard_dir = _candidate / "scripts"
    if (_guard_dir / "import_guard.py").exists():
        if str(_guard_dir) not in sys.path:
            sys.path.insert(0, str(_guard_dir))
        break
else:
    raise RuntimeError("Could not locate scripts/import_guard.py. Run this notebook from the WSAI workspace or copy the guard module alongside it.")

from import_guard import install_pandas_guards
install_pandas_guards()


In [ ]:
# Install dependencies
%pip -q install -U "sentence-transformers" "datasets" "accelerate" "transformers>=4.51.0,<5" "huggingface_hub" "tqdm" "pandas==2.2.2" "numpy==2.0.2" "scikit-learn>=1.5,<1.9" "matplotlib

## 1. Imports and project paths

For SSH / VS Code, run this notebook from `~/labse_all_pairs_indic_finetuning` with `USE_GOOGLE_DRIVE = False`.


In [ ]:
import os
import json
import math
import random
import shutil
import hashlib
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import batch_to_device
from transformers import get_linear_schedule_with_warmup

import matplotlib.pyplot as plt

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# -------------------------
# Project directory
# -------------------------
USE_GOOGLE_DRIVE = False

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_DIR = Path("/content/drive/MyDrive/labse_all_pairs_indic_finetuning")
else:
    PROJECT_DIR = Path.cwd().resolve()

OUTPUT_DIR = PROJECT_DIR / "outputs"
DATA_DIR = PROJECT_DIR / "data"

PHASE1_RUN_NAME = "labse_phase1_all462_1000pairs_balanced"
PHASE1_RUN_DIR = OUTPUT_DIR / PHASE1_RUN_NAME
PHASE1_BEST_MODEL_DIR = PHASE1_RUN_DIR / "best_model"
PHASE1_FINAL_MODEL_DIR = PHASE1_RUN_DIR / "final_model"

PHASE2_RUN_NAME = "labse_phase2_weak_pair_weighted_1000_from_phase1_1000"
PHASE2_RUN_DIR = OUTPUT_DIR / PHASE2_RUN_NAME
PHASE2_BEST_MODEL_DIR = PHASE2_RUN_DIR / "best_model"
PHASE2_FINAL_MODEL_DIR = PHASE2_RUN_DIR / "final_model"
PHASE2_CHECKPOINT_DIR = PHASE2_RUN_DIR / "checkpoints"
PHASE2_METRICS_DIR = PROJECT_DIR / "metrics" / PHASE2_RUN_NAME

for d in [OUTPUT_DIR, DATA_DIR, PHASE2_RUN_DIR, PHASE2_CHECKPOINT_DIR, PHASE2_METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project dir:", PROJECT_DIR)
print("Phase 1 best model:", PHASE1_BEST_MODEL_DIR)
print("Phase 2 run dir:", PHASE2_RUN_DIR)
print("Phase 2 metrics dir:", PHASE2_METRICS_DIR)
print("Phase 2 checkpoint dir:", PHASE2_CHECKPOINT_DIR)

assert PHASE1_BEST_MODEL_DIR.exists(), f"Phase 1 1000 best_model not found: {PHASE1_BEST_MODEL_DIR}"
assert (PHASE1_BEST_MODEL_DIR / "modules.json").exists(), "Phase 1 1000 best_model does not look like a SentenceTransformer folder"


## 2. Configuration

Phase 2 starts from the Phase 1 best model and uses a conservative continuation setup.

Recommended first run:

- `EPOCHS = 2`
- `LEARNING_RATE = 1e-6`
- `MAX_PAIR_WEIGHT = 3.0`

This gently biases training toward weak pairs without aggressively changing the Phase 1 embedding space.


In [ ]:
# Data
IN22_GEN_NAME = "ai4bharat/IN22-Gen"
IN22_CONV_NAME = "ai4bharat/IN22-Conv"
DATASET_CONFIG = "default"
SPLIT = "test"

# 1000 examples per directed pair:
# 462 × 1000 = 462,000 rows before train/validation split.
EXAMPLES_PER_DIRECTED_PAIR = 1000
VAL_SIZE = 0.10

# Training continuation from Phase 1 1000 model
MAX_SEQ_LENGTH = 128
BATCH_SIZE = 512
EPOCHS = 2
LEARNING_RATE = 5e-7
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0
MNRL_SCALE = 20.0
USE_AMP = torch.cuda.is_available()

# Weak-pair weighting
USE_WEIGHTED_SAMPLER = True
WEIGHT_SCORE_MODE = "composite"   # options: "composite", "cosine_gap", "accuracy_at_1", "balanced_accuracy"
MIN_PAIR_WEIGHT = 1.0
MAX_PAIR_WEIGHT = 3.0
WEIGHT_ALPHA = 1.0
EPS_SCORE = 1e-4

# Best model selection guard
SPECIFICITY_PENALTY_WEIGHT = 2.0

# Checkpointing
RESUME_FROM_CHECKPOINT = True
CHECKPOINT_KEEP_LAST = 2
SAVE_CHECKPOINT_EVERY_EPOCH = True
SKIP_IF_FINAL_MODEL_EXISTS = False

# Final IN22-Conv evaluation after training can be long. Keep True for final report.
RUN_FINAL_IN22_CONV_EVAL = True
IN22_CONV_EVAL_BATCH_SIZE = 256 if torch.cuda.is_available() else 64

print("Phase 2 1000 config")
print("Examples per directed pair:", EXAMPLES_PER_DIRECTED_PAIR)
print("Epochs:", EPOCHS)
print("LR:", LEARNING_RATE)
print("Batch size:", BATCH_SIZE)
print("Weighted sampler:", USE_WEIGHTED_SAMPLER)
print("Max pair weight:", MAX_PAIR_WEIGHT)
print("USE_AMP:", USE_AMP)


## 3. Language map and directed pair generation

There are 22 Indic languages. Directed pairs exclude same-language pairs:

\[
22 	imes 21 = 462
\]


In [ ]:
INDIC_LANGS = {
    "asm": "asm_Beng",
    "ben": "ben_Beng",
    "brx": "brx_Deva",
    "doi": "doi_Deva",
    "guj": "guj_Gujr",
    "hin": "hin_Deva",
    "kan": "kan_Knda",
    "kas": "kas_Arab",
    "gom": "gom_Deva",
    "mai": "mai_Deva",
    "mal": "mal_Mlym",
    "mni": "mni_Mtei",
    "mar": "mar_Deva",
    "npi": "npi_Deva",
    "ory": "ory_Orya",
    "pan": "pan_Guru",
    "san": "san_Deva",
    "sat": "sat_Olck",
    "snd": "snd_Deva",
    "tam": "tam_Taml",
    "tel": "tel_Telu",
    "urd": "urd_Arab",
}

LANG_CODES = list(INDIC_LANGS.keys())
DIRECTED_PAIRS = [(src, tgt) for src in LANG_CODES for tgt in LANG_CODES if src != tgt]

print("Number of Indic languages:", len(LANG_CODES))
print("Number of directed Indic-Indic pairs:", len(DIRECTED_PAIRS))
assert len(DIRECTED_PAIRS) == 462

print("First 10 pairs:", DIRECTED_PAIRS[:10])

## 4. Load IN22-Gen and construct balanced train/validation pairs

This uses IN22-Gen for training/validation. Each directed pair gets `EXAMPLES_PER_DIRECTED_PAIR` rows before the 90/10 split.


In [ ]:
def stable_int(text: str, modulo: int = 10**9) -> int:
    return int(hashlib.md5(text.encode("utf-8")).hexdigest(), 16) % modulo


def load_in22_dataframe(dataset_name: str) -> pd.DataFrame:
    ds = load_dataset(dataset_name, DATASET_CONFIG, split=SPLIT)
    df = pd.DataFrame(ds)
    print(dataset_name, "shape:", df.shape)
    return df


def validate_language_columns(df: pd.DataFrame, lang_map: Dict[str, str]):
    missing = [col for col in lang_map.values() if col not in df.columns]
    if missing:
        raise ValueError(f"Missing expected language columns: {missing}")
    print("All language columns present.")


def clean_text_series(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip()


def build_all_directed_pairs_df(
    df: pd.DataFrame,
    directed_pairs: List[Tuple[str, str]],
    examples_per_pair: int,
    lang_map: Dict[str, str],
    seed: int = 42,
) -> pd.DataFrame:
    rows = []
    for src, tgt in tqdm(directed_pairs, desc="Building directed-pair rows"):
        src_col = lang_map[src]
        tgt_col = lang_map[tgt]
        tmp = df[[src_col, tgt_col]].copy()
        tmp[src_col] = clean_text_series(tmp[src_col])
        tmp[tgt_col] = clean_text_series(tmp[tgt_col])
        tmp = tmp[(tmp[src_col] != "") & (tmp[tgt_col] != "")]
        tmp = tmp.dropna()

        if len(tmp) == 0:
            raise ValueError(f"No usable rows for {src}->{tgt}")

        n = min(examples_per_pair, len(tmp))
        pair_seed = seed + stable_int(f"{src}->{tgt}", 100000)
        tmp = tmp.sample(n=n, random_state=pair_seed).reset_index(drop=True)
        tmp = tmp.rename(columns={src_col: "sentence1", tgt_col: "sentence2"})
        tmp["src_lang"] = src
        tmp["tgt_lang"] = tgt
        tmp["direction"] = f"{src}->{tgt}"
        tmp["pair_row_id"] = np.arange(len(tmp))
        rows.append(tmp[["src_lang", "tgt_lang", "direction", "pair_row_id", "sentence1", "sentence2"]])

    out = pd.concat(rows, ignore_index=True)
    return out


def split_train_val_per_direction(pair_df: pd.DataFrame, val_size: float, seed: int = 42):
    train_parts = []
    val_parts = []
    for direction, group in tqdm(pair_df.groupby("direction", sort=False), desc="Splitting train/val"):
        g = group.sample(frac=1.0, random_state=seed + stable_int(direction, 100000)).reset_index(drop=True)
        val_n = max(1, int(round(len(g) * val_size)))
        val_parts.append(g.iloc[:val_n].copy())
        train_parts.append(g.iloc[val_n:].copy())

    train_df = pd.concat(train_parts, ignore_index=True)
    val_df = pd.concat(val_parts, ignore_index=True)
    return train_df, val_df

in22_gen_df = load_in22_dataframe(IN22_GEN_NAME)
validate_language_columns(in22_gen_df, INDIC_LANGS)

all_pairs_df = build_all_directed_pairs_df(
    in22_gen_df,
    DIRECTED_PAIRS,
    EXAMPLES_PER_DIRECTED_PAIR,
    INDIC_LANGS,
    seed=SEED,
)

train_df, val_df = split_train_val_per_direction(all_pairs_df, VAL_SIZE, seed=SEED)

print("All pair rows:", len(all_pairs_df))
print("Train rows:", len(train_df))
print("Val rows:", len(val_df))
print("Train directions:", train_df["direction"].nunique())
print("Val directions:", val_df["direction"].nunique())

assert all_pairs_df["direction"].nunique() == 462
assert train_df["direction"].nunique() == 462
assert val_df["direction"].nunique() == 462

# Save reproducibility files
train_pairs_path = PHASE2_METRICS_DIR / "phase2_train_pairs_used.csv"
val_pairs_path = PHASE2_METRICS_DIR / "phase2_val_pairs_used.csv"
all_pairs_path = PHASE2_METRICS_DIR / "phase2_all_pairs_before_split.csv"

all_pairs_df.to_csv(all_pairs_path, index=False)
train_df.to_csv(train_pairs_path, index=False)
val_df.to_csv(val_pairs_path, index=False)

print("Saved:", train_pairs_path)
print("Saved:", val_pairs_path)

## 5. Pair-level validation metrics

We first evaluate the Phase 1 model on the balanced IN22-Gen validation split. These pair-level scores are used to identify weak directions and compute sampling weights.

The final unseen benchmark remains IN22-Conv. We do not use IN22-Conv to create training weights.


In [ ]:
def deterministic_shift(direction: str, n: int) -> int:
    if n <= 1:
        return 0
    return 1 + stable_int(direction, n - 1)


def pair_metrics_from_embeddings(src_emb: np.ndarray, tgt_emb: np.ndarray, direction: str) -> Dict[str, float]:
    n = src_emb.shape[0]
    gold = np.sum(src_emb * tgt_emb, axis=1)

    if n > 1:
        shift = deterministic_shift(direction, n)
        rand_tgt = np.roll(tgt_emb, shift=shift, axis=0)
        random_cos = np.sum(src_emb * rand_tgt, axis=1)
    else:
        random_cos = np.array([0.0])

    mean_gold = float(np.mean(gold))
    mean_random = float(np.mean(random_cos))
    cosine_gap = mean_gold - mean_random
    threshold = (mean_gold + mean_random) / 2.0

    sensitivity = float(np.mean(gold >= threshold))
    specificity = float(np.mean(random_cos < threshold))
    balanced_accuracy = (sensitivity + specificity) / 2.0

    # Retrieval Accuracy@1 over this pair's validation rows
    sim = np.matmul(src_emb, tgt_emb.T)
    pred = np.argmax(sim, axis=1)
    accuracy_at_1 = float(np.mean(pred == np.arange(n)))

    # MRR / Recall small validation candidate set
    ranks = 1 + np.sum(sim > np.diag(sim)[:, None], axis=1)
    recall_at_5 = float(np.mean(ranks <= 5))
    recall_at_10 = float(np.mean(ranks <= 10))
    mrr = float(np.mean(1.0 / ranks))

    return {
        "n": int(n),
        "mean_gold_cosine": mean_gold,
        "std_gold_cosine": float(np.std(gold)),
        "mean_random_cosine": mean_random,
        "std_random_cosine": float(np.std(random_cos)),
        "cosine_gap": float(cosine_gap),
        "threshold_midpoint": float(threshold),
        "sensitivity_midpoint": sensitivity,
        "specificity_midpoint": specificity,
        "balanced_accuracy": float(balanced_accuracy),
        "accuracy_at_1": accuracy_at_1,
        "recall_at_5": recall_at_5,
        "recall_at_10": recall_at_10,
        "mrr": mrr,
    }


def evaluate_pair_level_on_dataframe(
    model: SentenceTransformer,
    eval_df: pd.DataFrame,
    batch_size: int = 128,
    desc: str = "Evaluating pair-level",
) -> pd.DataFrame:
    model.eval()
    rows = []
    for direction, group in tqdm(eval_df.groupby("direction", sort=False), desc=desc):
        src_lang = group["src_lang"].iloc[0]
        tgt_lang = group["tgt_lang"].iloc[0]
        src_texts = group["sentence1"].astype(str).tolist()
        tgt_texts = group["sentence2"].astype(str).tolist()

        src_emb = model.encode(
            src_texts,
            batch_size=batch_size,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )
        tgt_emb = model.encode(
            tgt_texts,
            batch_size=batch_size,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )

        metrics = pair_metrics_from_embeddings(src_emb, tgt_emb, direction)
        rows.append({
            "src_lang": src_lang,
            "tgt_lang": tgt_lang,
            "direction": direction,
            **metrics,
        })

    return pd.DataFrame(rows)


def summarize_pair_metrics(pair_metrics_df: pd.DataFrame) -> Dict[str, float]:
    # Weighted by n for row-level metrics.
    weights = pair_metrics_df["n"].values.astype(float)
    weights = weights / weights.sum()

    out = {}
    for col in [
        "mean_gold_cosine",
        "mean_random_cosine",
        "cosine_gap",
        "sensitivity_midpoint",
        "specificity_midpoint",
        "balanced_accuracy",
        "accuracy_at_1",
        "recall_at_5",
        "recall_at_10",
        "mrr",
    ]:
        out[col] = float(np.sum(pair_metrics_df[col].values * weights))
    out["num_pairs"] = int(len(pair_metrics_df))
    out["total_examples"] = int(pair_metrics_df["n"].sum())
    return out

phase1_model = SentenceTransformer(str(PHASE1_BEST_MODEL_DIR), device=DEVICE)
phase1_model.max_seq_length = MAX_SEQ_LENGTH

phase1_val_pair_metrics = evaluate_pair_level_on_dataframe(
    phase1_model,
    val_df,
    batch_size=IN22_CONV_EVAL_BATCH_SIZE,
    desc="Phase 1 model on IN22-Gen validation",
)

phase1_val_pair_metrics_path = PHASE2_METRICS_DIR / "phase2_pair_level_phase1_val_scores.csv"
phase1_val_pair_metrics.to_csv(phase1_val_pair_metrics_path, index=False)

phase1_val_summary = summarize_pair_metrics(phase1_val_pair_metrics)
phase1_val_summary_path = PHASE2_METRICS_DIR / "phase2_phase1_val_summary.json"
with open(phase1_val_summary_path, "w") as f:
    json.dump(phase1_val_summary, f, indent=2)

print("Phase 1 validation summary:")
print(json.dumps(phase1_val_summary, indent=2))
print("Saved pair scores:", phase1_val_pair_metrics_path)

# Release memory before training
try:
    del phase1_model
    torch.cuda.empty_cache()
except Exception:
    pass

## 6. Compute weak-pair weights

Weights are computed from Phase 1 validation pair-level scores.

For the first version, use a composite quality score:

\[
s_i = 0.5\,	ext{norm(cosine\_gap)} + 0.3\,	ext{norm(accuracy@1)} + 0.2\,	ext{norm(specificity)}
\]

Then:

\[
w_i \propto rac{1}{s_i}
\]

Weights are normalized and clipped to `[1.0, 3.0]` so weak pairs are sampled more often but cannot dominate training.


In [ ]:
def minmax_normalize(values: pd.Series, eps: float = 1e-8) -> pd.Series:
    v = values.astype(float)
    lo = v.min()
    hi = v.max()
    if abs(hi - lo) < eps:
        return pd.Series(np.ones(len(v)), index=v.index)
    return (v - lo) / (hi - lo)


def compute_pair_weights(pair_metrics_df: pd.DataFrame) -> pd.DataFrame:
    df = pair_metrics_df.copy()

    if WEIGHT_SCORE_MODE == "composite":
        df["norm_cosine_gap"] = minmax_normalize(df["cosine_gap"])
        df["norm_accuracy_at_1"] = minmax_normalize(df["accuracy_at_1"])
        df["norm_specificity"] = minmax_normalize(df["specificity_midpoint"])
        df["quality_score"] = (
            0.50 * df["norm_cosine_gap"]
            + 0.30 * df["norm_accuracy_at_1"]
            + 0.20 * df["norm_specificity"]
        )
    elif WEIGHT_SCORE_MODE == "cosine_gap":
        df["quality_score"] = minmax_normalize(df["cosine_gap"])
    elif WEIGHT_SCORE_MODE == "accuracy_at_1":
        df["quality_score"] = minmax_normalize(df["accuracy_at_1"])
    elif WEIGHT_SCORE_MODE == "balanced_accuracy":
        df["quality_score"] = minmax_normalize(df["balanced_accuracy"])
    else:
        raise ValueError(f"Unknown WEIGHT_SCORE_MODE: {WEIGHT_SCORE_MODE}")

    # Avoid zero scores causing infinite weights.
    df["quality_score_clipped"] = df["quality_score"].clip(lower=EPS_SCORE)
    df["raw_inverse_weight"] = (1.0 / df["quality_score_clipped"]) ** WEIGHT_ALPHA

    # Normalize raw weights by median, then clip.
    median_raw = float(df["raw_inverse_weight"].median())
    df["pair_weight_unclipped"] = df["raw_inverse_weight"] / median_raw
    df["pair_weight"] = df["pair_weight_unclipped"].clip(lower=MIN_PAIR_WEIGHT, upper=MAX_PAIR_WEIGHT)

    # Higher weakness rank = weaker pair.
    df["weakness_rank"] = df["pair_weight"].rank(method="dense", ascending=False).astype(int)
    df = df.sort_values(["pair_weight", "quality_score"], ascending=[False, True]).reset_index(drop=True)
    return df

pair_weights_df = compute_pair_weights(phase1_val_pair_metrics)

pair_weights_path = PHASE2_METRICS_DIR / "phase2_pair_weights.csv"
weak_pairs_path = PHASE2_METRICS_DIR / "phase2_weak_pairs_identified_top50.csv"
pair_weights_df.to_csv(pair_weights_path, index=False)
pair_weights_df.head(50).to_csv(weak_pairs_path, index=False)

print("Saved weights:", pair_weights_path)
print("Saved weak pairs:", weak_pairs_path)
print("Weight distribution:")
print(pair_weights_df["pair_weight"].describe())
print()
print("Top 20 weakest pairs by training weight:")
display_cols = ["direction", "cosine_gap", "accuracy_at_1", "specificity_midpoint", "balanced_accuracy", "quality_score", "pair_weight"]
display(pair_weights_df[display_cols].head(20))

# Attach weights to training rows
weight_map = pair_weights_df.set_index("direction")["pair_weight"].to_dict()
score_map = pair_weights_df.set_index("direction")["quality_score"].to_dict()

train_df_weighted = train_df.copy()
train_df_weighted["pair_weight"] = train_df_weighted["direction"].map(weight_map).astype(float)
train_df_weighted["quality_score"] = train_df_weighted["direction"].map(score_map).astype(float)

weighted_train_path = PHASE2_METRICS_DIR / "phase2_train_pairs_with_weights.csv"
train_df_weighted.to_csv(weighted_train_path, index=False)
print("Saved weighted training rows:", weighted_train_path)
print("Training rows:", len(train_df_weighted))

## 7. Dataset, DataLoader, loss, checkpoint helpers


In [ ]:
class PairTextDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.sentence1 = df["sentence1"].astype(str).tolist()
        self.sentence2 = df["sentence2"].astype(str).tolist()
        self.directions = df["direction"].astype(str).tolist()

    def __len__(self):
        return len(self.sentence1)

    def __getitem__(self, idx):
        return self.sentence1[idx], self.sentence2[idx], self.directions[idx]


def make_dataloader(model: SentenceTransformer, df: pd.DataFrame, batch_size: int, weighted: bool = True):
    dataset = PairTextDataset(df)

    def collate_fn(batch):
        texts1 = [x[0] for x in batch]
        texts2 = [x[1] for x in batch]
        directions = [x[2] for x in batch]
        sentence_features = [model.tokenize(texts1), model.tokenize(texts2)]
        return sentence_features, directions

    if weighted:
        sample_weights = torch.tensor(df["pair_weight"].values, dtype=torch.double)
        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True,
        )
        return DataLoader(
            dataset,
            sampler=sampler,
            batch_size=batch_size,
            drop_last=True,
            collate_fn=collate_fn,
            num_workers=0,
        )

    return DataLoader(
        dataset,
        shuffle=True,
        batch_size=batch_size,
        drop_last=True,
        collate_fn=collate_fn,
        num_workers=0,
    )


def mnrl_loss_from_sentence_features(model: SentenceTransformer, sentence_features, scale: float = 20.0):
    features1 = batch_to_device(sentence_features[0], DEVICE)
    features2 = batch_to_device(sentence_features[1], DEVICE)

    emb1 = model(features1)["sentence_embedding"]
    emb2 = model(features2)["sentence_embedding"]

    emb1 = F.normalize(emb1, p=2, dim=1)
    emb2 = F.normalize(emb2, p=2, dim=1)

    scores = torch.matmul(emb1, emb2.T) * scale
    labels = torch.arange(scores.size(0), device=scores.device)
    loss = F.cross_entropy(scores, labels)
    return loss


def latest_checkpoint(checkpoint_dir: Path):
    ckpts = []
    for p in checkpoint_dir.glob("epoch_*"):
        state_path = p / "training_state.pt"
        model_path = p / "model"
        if p.is_dir() and state_path.exists() and (model_path / "modules.json").exists():
            ckpts.append(p)
    if not ckpts:
        return None
    return sorted(ckpts, key=lambda x: int(x.name.split("_")[-1]))[-1]


def cleanup_old_checkpoints(checkpoint_dir: Path, keep_last: int):
    ckpts = []
    for p in checkpoint_dir.glob("epoch_*"):
        if p.is_dir():
            try:
                epoch_num = int(p.name.split("_")[-1])
            except Exception:
                continue
            ckpts.append((epoch_num, p))
    ckpts = sorted(ckpts, key=lambda x: x[0])
    if len(ckpts) <= keep_last:
        return
    for _, p in ckpts[:-keep_last]:
        print("Removing old checkpoint:", p)
        shutil.rmtree(p, ignore_errors=True)


def save_epoch_checkpoint(
    epoch: int,
    model: SentenceTransformer,
    optimizer,
    scheduler,
    scaler,
    global_step: int,
    best_score: float,
    train_history: List[Dict],
):
    ckpt_dir = PHASE2_CHECKPOINT_DIR / f"epoch_{epoch:03d}"
    model_dir = ckpt_dir / "model"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    model.save(str(model_dir))
    state = {
        "epoch": epoch,
        "global_step": global_step,
        "best_score": best_score,
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict() if scaler is not None else None,
        "config": TRAINING_CONFIG,
    }
    torch.save(state, ckpt_dir / "training_state.pt")
    pd.DataFrame(train_history).to_csv(ckpt_dir / "train_metrics_until_checkpoint.csv", index=False)
    cleanup_old_checkpoints(PHASE2_CHECKPOINT_DIR, CHECKPOINT_KEEP_LAST)
    print("Saved checkpoint:", ckpt_dir)


def load_checkpoint_or_phase1():
    ckpt = latest_checkpoint(PHASE2_CHECKPOINT_DIR) if RESUME_FROM_CHECKPOINT else None
    if ckpt is None:
        print("No Phase 2 checkpoint found. Starting from Phase 1 best_model.")
        model = SentenceTransformer(str(PHASE1_BEST_MODEL_DIR), device=DEVICE)
        model.max_seq_length = MAX_SEQ_LENGTH
        return model, None, 1, 0, -1e9, []

    print("Loading Phase 2 checkpoint:", ckpt)
    model = SentenceTransformer(str(ckpt / "model"), device=DEVICE)
    model.max_seq_length = MAX_SEQ_LENGTH
    state = torch.load(ckpt / "training_state.pt", map_location=DEVICE)
    start_epoch = int(state["epoch"]) + 1
    global_step = int(state.get("global_step", 0))
    best_score = float(state.get("best_score", -1e9))

    hist_path = ckpt / "train_metrics_until_checkpoint.csv"
    if hist_path.exists():
        history = pd.read_csv(hist_path).to_dict("records")
    else:
        history = []

    return model, state, start_epoch, global_step, best_score, history


def save_training_config():
    PHASE2_RUN_DIR.mkdir(parents=True, exist_ok=True)
    PHASE2_METRICS_DIR.mkdir(parents=True, exist_ok=True)
    path1 = PHASE2_RUN_DIR / "training_config.json"
    path2 = PHASE2_METRICS_DIR / "training_config.json"
    with open(path1, "w") as f:
        json.dump(TRAINING_CONFIG, f, indent=2, default=str)
    with open(path2, "w") as f:
        json.dump(TRAINING_CONFIG, f, indent=2, default=str)
    print("Saved config:", path1)

TRAINING_CONFIG = {
    "phase": "Phase 2 weak-pair weighted fine-tuning",
    "init_model": str(PHASE1_BEST_MODEL_DIR),
    "project_dir": str(PROJECT_DIR),
    "phase2_run_dir": str(PHASE2_RUN_DIR),
    "training_pairs": "all 462 directed Indic-Indic pairs",
    "examples_per_directed_pair": EXAMPLES_PER_DIRECTED_PAIR,
    "total_train_rows": int(len(train_df_weighted)),
    "total_val_rows": int(len(val_df)),
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "max_seq_length": MAX_SEQ_LENGTH,
    "weighted_sampler": USE_WEIGHTED_SAMPLER,
    "weight_score_mode": WEIGHT_SCORE_MODE,
    "min_pair_weight": MIN_PAIR_WEIGHT,
    "max_pair_weight": MAX_PAIR_WEIGHT,
    "weight_alpha": WEIGHT_ALPHA,
    "mnrl_scale": MNRL_SCALE,
    "specificity_penalty_weight": SPECIFICITY_PENALTY_WEIGHT,
    "phase1_val_summary": phase1_val_summary,
}

save_training_config()

## 8. Train Phase 2 weak-pair weighted model

This cell supports full epoch-level checkpoint resume. If runtime disconnects after an epoch completes, rerunning will resume from the next epoch.


In [ ]:
if SKIP_IF_FINAL_MODEL_EXISTS and (PHASE2_FINAL_MODEL_DIR / "modules.json").exists():
    print("Final model already exists and SKIP_IF_FINAL_MODEL_EXISTS=True. Skipping training.")
else:
    model, loaded_state, start_epoch, global_step, best_score, train_history = load_checkpoint_or_phase1()
    model.to(DEVICE)
    model.max_seq_length = MAX_SEQ_LENGTH

    train_dataloader = make_dataloader(
        model=model,
        df=train_df_weighted,
        batch_size=BATCH_SIZE,
        weighted=USE_WEIGHTED_SAMPLER,
    )

    batches_per_epoch = len(train_dataloader)
    total_steps = batches_per_epoch * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    if loaded_state is not None:
        optimizer.load_state_dict(loaded_state["optimizer_state"])
        scheduler.load_state_dict(loaded_state["scheduler_state"])
        if loaded_state.get("scaler_state") is not None:
            scaler.load_state_dict(loaded_state["scaler_state"])
        print("Restored optimizer/scheduler/scaler from checkpoint.")

    print("Training rows:", len(train_df_weighted))
    print("Batches per epoch:", batches_per_epoch)
    print("Total planned steps:", total_steps)
    print("Warmup steps:", warmup_steps)
    print("Start epoch:", start_epoch)
    print("Current best score:", best_score)

    phase1_val_specificity = float(phase1_val_summary["specificity_midpoint"])

    for epoch in range(start_epoch, EPOCHS + 1):
        model.train()
        epoch_losses = []

        progress = tqdm(train_dataloader, desc=f"{PHASE2_RUN_NAME} | epoch {epoch}/{EPOCHS}")
        for sentence_features, directions in progress:
            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                loss = mnrl_loss_from_sentence_features(model, sentence_features, scale=MNRL_SCALE)

            if not torch.isfinite(loss):
                print("Non-finite loss detected. Skipping batch.")
                continue

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            global_step += 1
            loss_float = float(loss.detach().cpu().item())
            epoch_losses.append(loss_float)
            progress.set_postfix({
                "loss": f"{loss_float:.4f}",
                "lr": f"{scheduler.get_last_lr()[0]:.2e}",
            })

        mean_train_loss = float(np.mean(epoch_losses)) if epoch_losses else float("nan")

        # Balanced validation after every epoch.
        val_pair_metrics = evaluate_pair_level_on_dataframe(
            model,
            val_df,
            batch_size=IN22_CONV_EVAL_BATCH_SIZE,
            desc=f"Validation after epoch {epoch}",
        )
        val_pair_metrics.to_csv(PHASE2_METRICS_DIR / f"phase2_val_pair_metrics_epoch_{epoch:03d}.csv", index=False)
        val_summary = summarize_pair_metrics(val_pair_metrics)

        specificity_drop = max(0.0, phase1_val_specificity - val_summary["specificity_midpoint"])
        guarded_score = val_summary["cosine_gap"] - SPECIFICITY_PENALTY_WEIGHT * specificity_drop

        row = {
            "epoch": epoch,
            "global_step": global_step,
            "mean_train_loss": mean_train_loss,
            "val_guarded_score": float(guarded_score),
            "specificity_drop_vs_phase1_val": float(specificity_drop),
            **{f"val_{k}": v for k, v in val_summary.items()},
        }
        train_history.append(row)

        metrics_df = pd.DataFrame(train_history)
        metrics_df.to_csv(PHASE2_RUN_DIR / "phase2_train_metrics.csv", index=False)
        metrics_df.to_csv(PHASE2_METRICS_DIR / "phase2_train_metrics.csv", index=False)

        print("Epoch summary:")
        print(json.dumps(row, indent=2))

        if guarded_score > best_score:
            best_score = float(guarded_score)
            if PHASE2_BEST_MODEL_DIR.exists():
                shutil.rmtree(PHASE2_BEST_MODEL_DIR)
            model.save(str(PHASE2_BEST_MODEL_DIR))
            print("Saved new best model:", PHASE2_BEST_MODEL_DIR)

        if SAVE_CHECKPOINT_EVERY_EPOCH:
            save_epoch_checkpoint(
                epoch=epoch,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=scaler,
                global_step=global_step,
                best_score=best_score,
                train_history=train_history,
            )

    if PHASE2_FINAL_MODEL_DIR.exists():
        shutil.rmtree(PHASE2_FINAL_MODEL_DIR)
    model.save(str(PHASE2_FINAL_MODEL_DIR))
    print("Saved final model:", PHASE2_FINAL_MODEL_DIR)

    # Save final verification
    verification = {
        "best_model_exists": (PHASE2_BEST_MODEL_DIR / "modules.json").exists(),
        "final_model_exists": (PHASE2_FINAL_MODEL_DIR / "modules.json").exists(),
        "train_metrics_exists": (PHASE2_METRICS_DIR / "phase2_train_metrics.csv").exists(),
        "pair_weights_exists": pair_weights_path.exists(),
        "best_score": best_score,
        "global_step": global_step,
    }
    with open(PHASE2_METRICS_DIR / "phase2_training_verification.json", "w") as f:
        json.dump(verification, f, indent=2)
    print("Verification:")
    print(json.dumps(verification, indent=2))

## 9. Optional final IN22-Conv evaluation

This compares Phase 1 and Phase 2 on unseen IN22-Conv across all 462 directed pairs.

This can take time, but it produces the final evidence for Phase 2.


In [ ]:
def encode_language_cache(model: SentenceTransformer, df: pd.DataFrame, lang_map: Dict[str, str], batch_size: int = 128):
    cache = {}
    model.eval()
    for lang, col in tqdm(lang_map.items(), desc="Encoding all languages"):
        texts = df[col].astype(str).str.strip().tolist()
        cache[lang] = model.encode(
            texts,
            batch_size=batch_size,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )
    return cache


def metrics_from_full_retrieval(src_emb: np.ndarray, tgt_emb: np.ndarray, direction: str):
    n = src_emb.shape[0]
    sim = np.matmul(src_emb, tgt_emb.T)
    gold = np.diag(sim)

    ranks = 1 + np.sum(sim > gold[:, None], axis=1)
    accuracy_at_1 = float(np.mean(ranks == 1))
    recall_at_5 = float(np.mean(ranks <= 5))
    recall_at_10 = float(np.mean(ranks <= 10))
    mrr = float(np.mean(1.0 / ranks))
    mean_rank = float(np.mean(ranks))
    median_rank = float(np.median(ranks))

    if n > 1:
        shift = deterministic_shift(direction, n)
        random_cos = np.sum(src_emb * np.roll(tgt_emb, shift=shift, axis=0), axis=1)
    else:
        random_cos = np.array([0.0])

    mean_gold = float(np.mean(gold))
    mean_random = float(np.mean(random_cos))
    cosine_gap = mean_gold - mean_random
    threshold = (mean_gold + mean_random) / 2.0
    sensitivity = float(np.mean(gold >= threshold))
    specificity = float(np.mean(random_cos < threshold))
    balanced_accuracy = (sensitivity + specificity) / 2.0

    return {
        "n": int(n),
        "accuracy_at_1": accuracy_at_1,
        "recall_at_5": recall_at_5,
        "recall_at_10": recall_at_10,
        "mrr": mrr,
        "mean_rank": mean_rank,
        "median_rank": median_rank,
        "mean_gold_cosine": mean_gold,
        "std_gold_cosine": float(np.std(gold)),
        "mean_random_cosine": mean_random,
        "std_random_cosine": float(np.std(random_cos)),
        "cosine_gap": float(cosine_gap),
        "threshold_midpoint": float(threshold),
        "sensitivity_midpoint": sensitivity,
        "specificity_midpoint": specificity,
        "balanced_accuracy": float(balanced_accuracy),
    }


def evaluate_model_on_in22_conv(model_name: str, model_path: Path, in22_conv_df: pd.DataFrame) -> pd.DataFrame:
    print("Loading model for IN22-Conv eval:", model_name, model_path)
    model = SentenceTransformer(str(model_path), device=DEVICE)
    model.max_seq_length = MAX_SEQ_LENGTH

    emb_cache = encode_language_cache(model, in22_conv_df, INDIC_LANGS, batch_size=IN22_CONV_EVAL_BATCH_SIZE)

    rows = []
    for src, tgt in tqdm(DIRECTED_PAIRS, desc=f"Pair eval: {model_name}"):
        direction = f"{src}->{tgt}"
        metrics = metrics_from_full_retrieval(emb_cache[src], emb_cache[tgt], direction)
        rows.append({
            "model": model_name,
            "src_lang": src,
            "tgt_lang": tgt,
            "direction": direction,
            **metrics,
        })

    try:
        del model
        del emb_cache
        torch.cuda.empty_cache()
    except Exception:
        pass

    return pd.DataFrame(rows)


def summarize_eval_results(eval_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for model_name, g in eval_df.groupby("model", sort=False):
        weights = g["n"].values.astype(float)
        weights = weights / weights.sum()
        row = {"model": model_name, "num_pairs": len(g), "total_examples": int(g["n"].sum())}
        for col in [
            "accuracy_at_1",
            "recall_at_5",
            "recall_at_10",
            "mrr",
            "mean_rank",
            "median_rank",
            "mean_gold_cosine",
            "mean_random_cosine",
            "cosine_gap",
            "sensitivity_midpoint",
            "specificity_midpoint",
            "balanced_accuracy",
        ]:
            row[col] = float(np.sum(g[col].values * weights))
        rows.append(row)
    return pd.DataFrame(rows)

if RUN_FINAL_IN22_CONV_EVAL:
    in22_conv_df = load_in22_dataframe(IN22_CONV_NAME)
    validate_language_columns(in22_conv_df, INDIC_LANGS)

    model_paths = {
        "phase1_balanced_best": PHASE1_BEST_MODEL_DIR,
        "phase2_weighted_best": PHASE2_BEST_MODEL_DIR,
        "phase2_weighted_final": PHASE2_FINAL_MODEL_DIR,
    }

    eval_parts = []
    for model_name, model_path in model_paths.items():
        if not (model_path / "modules.json").exists():
            print("Skipping missing model:", model_name, model_path)
            continue
        part = evaluate_model_on_in22_conv(model_name, model_path, in22_conv_df)
        part.to_csv(PHASE2_METRICS_DIR / f"in22_conv_eval_{model_name}_by_pair.csv", index=False)
        eval_parts.append(part)

    all_eval = pd.concat(eval_parts, ignore_index=True)
    all_eval_path = PHASE2_METRICS_DIR / "phase2_in22_conv_eval_all_models_by_pair.csv"
    all_eval.to_csv(all_eval_path, index=False)

    summary = summarize_eval_results(all_eval)
    summary_path = PHASE2_METRICS_DIR / "phase2_in22_conv_eval_summary.csv"
    summary.to_csv(summary_path, index=False)

    print("Saved all-pair eval:", all_eval_path)
    print("Saved summary:", summary_path)
    display(summary)

    # Delta vs Phase 1 by directed pair for Phase 2 best.
    phase1 = all_eval[all_eval["model"] == "phase1_balanced_best"].copy()
    phase2 = all_eval[all_eval["model"] == "phase2_weighted_best"].copy()
    if len(phase1) and len(phase2):
        delta = phase2.merge(
            phase1,
            on=["src_lang", "tgt_lang", "direction"],
            suffixes=("_phase2", "_phase1"),
        )
        for col in ["accuracy_at_1", "cosine_gap", "sensitivity_midpoint", "specificity_midpoint", "balanced_accuracy"]:
            delta[f"delta_{col}"] = delta[f"{col}_phase2"] - delta[f"{col}_phase1"]

        delta_path = PHASE2_METRICS_DIR / "phase2_delta_vs_phase1_by_pair.csv"
        delta.to_csv(delta_path, index=False)
        print("Saved delta vs Phase 1:", delta_path)

        print("Top 20 improved weak/weighted pairs by delta cosine gap:")
        display(delta.sort_values("delta_cosine_gap", ascending=False).head(20)[[
            "direction", "cosine_gap_phase1", "cosine_gap_phase2", "delta_cosine_gap",
            "accuracy_at_1_phase1", "accuracy_at_1_phase2", "delta_accuracy_at_1",
            "specificity_midpoint_phase1", "specificity_midpoint_phase2", "delta_specificity_midpoint"
        ]])

        print("Top 20 degraded pairs by delta cosine gap:")
        display(delta.sort_values("delta_cosine_gap", ascending=True).head(20)[[
            "direction", "cosine_gap_phase1", "cosine_gap_phase2", "delta_cosine_gap",
            "accuracy_at_1_phase1", "accuracy_at_1_phase2", "delta_accuracy_at_1",
            "specificity_midpoint_phase1", "specificity_midpoint_phase2", "delta_specificity_midpoint"
        ]])
else:
    print("RUN_FINAL_IN22_CONV_EVAL is False. Skipping final evaluation.")

## 10. Quick verification and output listing


In [ ]:
print("Phase 2 best model:", PHASE2_BEST_MODEL_DIR, (PHASE2_BEST_MODEL_DIR / "modules.json").exists())
print("Phase 2 final model:", PHASE2_FINAL_MODEL_DIR, (PHASE2_FINAL_MODEL_DIR / "modules.json").exists())
print("Metrics dir:", PHASE2_METRICS_DIR)
print()
print("Files in metrics dir:")
for p in sorted(PHASE2_METRICS_DIR.glob("*")):
    size_mb = p.stat().st_size / (1024 ** 2) if p.is_file() else 0
    print(f"{p.name:70s} {size_mb:8.2f} MB")

## Notes for interpretation

Phase 2 is successful if it improves weak-pair results while preserving the global gains from Phase 1.

Compare against Phase 1:

- Overall Accuracy@1 should stay close to or above Phase 1.
- Overall cosine gap should stay close to or above Phase 1.
- Specificity should not collapse.
- Weak-pair average should improve.
- Strong pairs should not degrade heavily.

If Phase 2 improves weak pairs but specificity drops sharply, move to Phase 3: weak-pair weighting + LaBSE preservation/distillation loss.
